# Exercise 02 — Train agents in classic MDPs
**Mathematics of Reinforcement Learning**

In JupyterLite, use **Python (Pyodide)** and run the cells from top to bottom. In Acrobot, run the Q-learning experiment and its replay before moving on to policy search.


### Q-learning

The action value $Q^\pi(s,a)$ is the expected discounted return when we take action $a$ in state $s$ and follow policy $\pi$ from the next step onward:
$$
Q^\pi(s,a)=\mathbb{E}\!\left[\sum_{t=0}^{\infty}\gamma^t R_{t+1}
\,\middle|\, S_0=s,\ A_0=a\right],
$$
where the continuation uses $\pi$. The Q-table estimates the optimal action values $Q^*(s,a)=\sup_\pi Q^\pi(s,a)$.

After observing $(s,a,r,s')$, update the entry for $(s,a)$:
$$
Q(s,a)\leftarrow Q(s,a)+\alpha\bigl(r+\gamma\max_b Q(s',b)-Q(s,a)\bigr),
$$
During training, choose a random action with probability $\varepsilon$ and a greedy action otherwise. During evaluation, choose greedy actions and report the undiscounted sum of rewards.


In [ ]:
import math
from random import Random
import matplotlib.pyplot as plt

ACTIONS = [(0, -1), (1, 0), (0, 1), (-1, 0)]  # left, down, right, up
ARROWS = ['←', '↓', '→', '↑']

class Grid:
    def __init__(self, rows, slippery=False, limit=100):
        self.rows = rows
        self.height, self.width = len(rows), len(rows[0])
        self.start = ''.join(rows).index('S')
        self.slippery, self.limit = slippery, limit

    def reset(self, rng):
        return self.start

    def step(self, state, action, rng):
        if self.slippery:
            action = (action + rng.choice([-1, 0, 1])) % 4
        r, c = divmod(state, self.width)
        dr, dc = ACTIONS[action]
        r = max(0, min(self.height - 1, r + dr))
        c = max(0, min(self.width - 1, c + dc))
        tile = self.rows[r][c]
        reward = int(tile == 'G')
        return r * self.width + c, reward, tile in 'HG'

def episode(env, policy, seed):
    rng = Random(seed)
    s = env.reset(rng)
    path, total = [s], 0
    for _ in range(env.limit):
        s, reward, done = env.step(s, policy(s, rng), rng)
        path.append(s)
        total += reward
        if done:
            break
    return path, total

def compare(env, baseline, learned, seeds):
    for name, policy in [('Random', baseline), ('Trained', learned)]:
        returns = [episode(env, policy, seed)[1] for seed in seeds]
        print(f'{name:8} mean return over {len(returns)} episodes: {sum(returns)/len(returns):.2f}')

def greedy(row, rng):
    best = max(row)
    return rng.choice([a for a, value in enumerate(row) if value == best])

def train_grid(env, episodes, epsilon, seed=7):
    rng = Random(seed)
    q = [[0.0] * 4 for _ in range(env.height * env.width)]
    for _ in range(episodes):
        s = env.reset(rng)
        for _ in range(env.limit):
            a = rng.randrange(4) if rng.random() < epsilon else greedy(q[s], rng)
            nxt, reward, done = env.step(s, a, rng)
            target = reward if done else reward + 0.95 * max(q[nxt])
            q[s][a] += 0.15 * (target - q[s][a])
            s = nxt
            if done:
                break
    return q

def show_grid(env, q, seed):
    policy = lambda s, rng: greedy(q[s], rng)
    compare(env, lambda s, rng: rng.randrange(4), policy, range(1000, 1100))
    path, total = episode(env, policy, seed)
    fig, axes = plt.subplots(1, 2, figsize=(min(12, env.width + 3), 3))
    for ax, title in zip(axes, ['Greedy actions', f'One episode · return {total}']):
        for r, row in enumerate(env.rows):
            for c, tile in enumerate(row):
                s = r * env.width + c
                label = tile if tile in 'HG' else ''
                if ax == axes[0] and not label:
                    best = max(q[s])
                    label = ''.join(ARROWS[a] for a in range(4) if q[s][a] == best)
                elif tile == 'S':
                    label = 'S'
                ax.text(c, r, label, ha='center', va='center', fontsize=9)
        ax.set(xlim=(-0.5, env.width-0.5), ylim=(env.height-0.5, -0.5), aspect='equal', title=title)
        ax.set_xticks([i-0.5 for i in range(env.width+1)])
        ax.set_yticks([i-0.5 for i in range(env.height+1)])
        ax.tick_params(labelleft=False, labelbottom=False, length=0)
        ax.grid(alpha=0.3)
    axes[1].plot([s % env.width for s in path], [s // env.width for s in path],
                 'o-', color='teal', markersize=3, alpha=0.55)
    plt.tight_layout()
    plt.show()


## 1. FrozenLake

Cross a frozen lake from **S** to **G** without falling into a hole **H**. On slippery ice, the agent can slide sideways even when it chooses to move straight.

- **State space.** $\mathcal S=\{0,1,2,3\}^2$. Each state is a tile specified by its row and column. The agent starts on **S**. **G** and **H** are terminal.
- **Action space.** $\mathcal A=\{\text{left},\text{down},\text{right},\text{up}\}$.
- **Transitions and reward.** With slipping enabled, the intended direction and the two perpendicular directions have probability $1/3$ each. A move against a wall leaves the agent in place. Entering **G** gives $+1$. Every other transition gives $0$. A rollout stops after at most 100 steps.

The mean return is the fraction of evaluation episodes that reach **G**. The arrows show the greedy actions, including ties.


In [ ]:
lake = Grid(['SFFF', 'FHFH', 'FFFH', 'HFFG'], slippery=True)
lake_episodes = 3
lake_q = train_grid(lake, episodes=lake_episodes, epsilon=0.25)
show_grid(lake, lake_q, seed=12)


## 2. CartPole

Balance a pole on a moving cart by pushing the cart left or right. Keep the pole upright without letting the cart roll off the track.

- **State space.** $s=(x,\dot x,\theta,\dot\theta)\in\mathbb R^4$. Here $x$ and $\dot x$ are the cart's position and velocity, and $\theta$ and $\dot\theta$ are the pole's angle from upright and its angular velocity. Each component starts uniformly in $[-0.05,0.05]$.
- **Action space.** $\mathcal A=\{\text{push left},\text{push right}\}$. The applied force is $-10$ or $+10$, respectively.
- **Transitions and reward.** The cart and pole evolve deterministically given the state and force. Every step earns $+1$, including the step that ends the episode. The episode terminates when $|x|>2.4$ or $|\theta|>12^\circ$. A rollout stops after at most 500 steps.

Divide each state coordinate into bins. States in the same combination of bins share a row of the Q-table. The mean return is the mean number of steps completed.


In [ ]:
class CartPole:
    actions = 2
    limit = 500
    def reset(self, rng):
        return tuple(rng.uniform(-0.05, 0.05) for _ in range(4))

    def step(self, state, action, rng):
        x, v, angle, omega = state
        sin, cos = math.sin(angle), math.cos(angle)
        force = 10 if action else -10
        temp = (force + 0.05 * omega**2 * sin) / 1.1
        angular_acc = (9.8 * sin - cos * temp) / (0.5 * (4/3 - 0.1 * cos**2 / 1.1))
        cart_acc = temp - 0.05 * angular_acc * cos / 1.1
        nxt = (x + 0.02*v, v + 0.02*cart_acc, angle + 0.02*omega, omega + 0.02*angular_acc)
        done = abs(nxt[0]) > 2.4 or abs(nxt[2]) > math.radians(12)
        return nxt, 1, done



from collections import defaultdict

def bucket(value, bounds, bins):
    low, high = bounds
    return min(bins - 1, max(0, int(bins * (value - low) / (high - low))))

def train_control_q(env, bounds, bins, *, seed, steps=None, episodes=None):
    if (steps is None) == (episodes is None):
        raise ValueError("Specify either steps or episodes.")
    rng = Random(seed)
    table = defaultdict(lambda: [0.0] * env.actions)

    def encode(state):
        return tuple(bucket(x, limits, n) for x, limits, n in zip(state, bounds, bins))

    def policy(state, rng):
        return greedy(table.get(encode(state), [0.0] * env.actions), rng)

    state = env.reset(rng)
    length = 0
    completed = 0
    used = 0
    while (used < steps if steps is not None else completed < episodes):
        key = encode(state)
        progress = used / steps if steps is not None else completed / episodes
        epsilon = max(0.05, 0.35 * (1 - progress))
        action = rng.randrange(env.actions) if rng.random() < epsilon else policy(state, rng)
        next_state, reward, done = env.step(state, action, rng)
        target = reward if done else reward + 0.99 * max(table[encode(next_state)])
        table[key][action] += 0.15 * (target - table[key][action])
        state = next_state
        length += 1
        used += 1
        if done or length == env.limit:
            completed += 1
            state = env.reset(rng)
            length = 0
    return policy, used

def evaluate(env, policy):
    returns = [episode(env, policy, seed)[1] for seed in range(1000, 1020)]
    return sum(returns) / len(returns)


def replay(env, policy, draw, title, bounds, *, seed=1000, stride=2, interval=150, target=None):
    from matplotlib.animation import FuncAnimation
    from IPython.display import HTML, display

    path, total = episode(env, policy, seed)
    fig, ax = plt.subplots(figsize=(5.3, 3.4))
    ax.set(xlim=bounds[0], ylim=bounds[1], aspect='equal', title=f'{title} | Return {total}')
    ax.axhline(0, color='0.8')
    if target is not None:
        ax.axhline(target, color='tab:orange', linestyle='--', label='Target height')
        ax.legend(loc='lower right')
    line, = ax.plot([], [], 'o-', lw=3, color='teal')
    caption = ax.text(0.03, 0.96, '', transform=ax.transAxes, va='top')
    last = len(path) - 1

    def frame(t):
        line.set_data(*draw(path[t]))
        caption.set_text(f'Step {t} / {last}')
        return line, caption

    frames = list(range(0, last, stride)) + [last]
    animation = FuncAnimation(fig, frame, frames=frames, interval=interval, repeat=False)
    plt.close(fig)
    display(HTML(animation.to_jshtml(default_mode='once')))


In [ ]:
cart = CartPole()
cart_seed = 7
cart_steps = 5_000
cart_bins = [3, 5, 8, 9]
cart_bounds = [(-2.4, 2.4), (-2, 2), (-0.21, 0.21), (-3, 3)]

cart_agent, _ = train_control_q(cart, cart_bounds, cart_bins, steps=cart_steps, seed=cart_seed)
print(f"Q-learning: {evaluate(cart, cart_agent):.1f} mean steps over 20 test episodes")


In [ ]:
def cart_points(state):
    x, _, angle, _ = state
    return [x, x + math.sin(angle)], [0, math.cos(angle)]


replay(cart, cart_agent, cart_points, 'CartPole · Q-learning',
       ((-2.8, 2.8), (-0.25, 1.2)), interval=60)


## 3. Acrobot

Swing a two-link arm from its hanging position until its tip rises above the target line. The upper joint is passive. The motor acts only at the joint between the links, so the agent builds up momentum by swinging.

- **State space.** $s=(\theta_1,\theta_2,\omega_1,\omega_2)\in[-\pi,\pi)^2\times[-4\pi,4\pi]\times[-9\pi,9\pi]$. The first angle is measured from vertically down and the second relative to the first link. The angular velocities are their time derivatives. Each component starts uniformly in $[-0.1,0.1]$.
- **Action space.** $\mathcal A=\{-1,0,+1\}$, the torque applied at the second joint.
- **Transitions and reward.** The arm follows deterministic dynamics. The tip height is $h(s)=-\cos\theta_1-\cos(\theta_1+\theta_2)$ relative to the fixed pivot. A transition earns $-1$ while $h(s')\leq 1$, and $0$ when $h(s')>1$, which ends the episode. A rollout stops after at most 500 steps.

### Tabular Q-learning

Train with eight bins per state coordinate. Play the replay and check whether the tip crosses the dashed target line. Each training run starts with a fresh Q-table.


In [ ]:
class Acrobot:
    actions = 3
    limit = 500

    def reset(self, rng):
        return tuple(rng.uniform(-0.1, 0.1) for _ in range(4))

    @staticmethod
    def derivative(state, torque):
        q1, q2, v1, v2 = state
        c, s = math.cos(q2), math.sin(q2)
        d1 = 3.5 + c
        d2 = 1.25 + 0.5 * c
        gravity2 = 4.9 * math.sin(q1 + q2)
        gravity1 = -0.5*s*(v2*v2 + 2*v1*v2) + 14.7*math.sin(q1) + gravity2
        a2 = (torque + d2 * gravity1 / d1 - 0.5*v1*v1*s - gravity2) / (1.25 - d2*d2/d1)
        a1 = -(d2*a2 + gravity1) / d1
        return v1, v2, a1, a2

    def step(self, state, action, rng):
        torque = action - 1
        k1 = self.derivative(state, torque)
        k2 = self.derivative(tuple(x + 0.1*k for x, k in zip(state, k1)), torque)
        k3 = self.derivative(tuple(x + 0.1*k for x, k in zip(state, k2)), torque)
        k4 = self.derivative(tuple(x + 0.2*k for x, k in zip(state, k3)), torque)
        a, b, u, v = (x + (k + 2*l + 2*m + n)/30
                      for x, k, l, m, n in zip(state, k1, k2, k3, k4))
        wrap = lambda angle: (angle + math.pi) % (2*math.pi) - math.pi
        clip = lambda value, limit: max(-limit, min(limit, value))
        next_state = wrap(a), wrap(b), clip(u, 4*math.pi), clip(v, 9*math.pi)
        done = -math.cos(a) - math.cos(a + b) > 1
        return next_state, 0 if done else -1, done

acro = Acrobot()
acro_bounds = [(-math.pi, math.pi), (-math.pi, math.pi),
               (-4*math.pi, 4*math.pi), (-9*math.pi, 9*math.pi)]
ACRO_TEST_SEEDS = range(1000, 1020)

def acro_points(state):
    a, b, _, _ = state
    x, y = math.sin(a), -math.cos(a)
    return [0, x, x + math.sin(a + b)], [0, y, y - math.cos(a + b)]

def acro_results(label, policy, training_steps):
    runs = [episode(acro, policy, seed) for seed in ACRO_TEST_SEEDS]
    mean_return = sum(total for _, total in runs) / len(runs)
    successes = sum(acro_points(path[-1])[1][-1] > 1 for path, _ in runs)
    print(f"{label}: mean return {mean_return:.1f} | "
          f"target reached in {successes}/{len(runs)} test episodes | "
          f"{training_steps:,} training steps")


In [ ]:
acro_seed = 7
acro_episodes = 1000
acro_bins = [8, 8, 8, 8]
acro_q, acro_q_steps = train_control_q(
    acro, acro_bounds, acro_bins, episodes=acro_episodes, seed=acro_seed
)
acro_results('Tabular Q-learning', acro_q, acro_q_steps)


In [ ]:
acro_replay_seed = 1015
replay(acro, acro_q, acro_points, 'Acrobot · Tabular Q-learning',
       ((-2.1, 2.1), (-2.1, 2.1)), seed=acro_replay_seed, target=1)


### Direct policy search

Use one weight vector $w$ to choose a torque from the continuous state. Compute $z=w^\top\phi(s)$, where $\phi(s)$ contains the sines and cosines of the two angles and the rescaled angular velocities. Apply $+1$ if $z>0.1$, $-1$ if $z<-0.1$, and $0$ otherwise.

Sample weight vectors, run each policy from three training starts, and score it by its mean return. Sample the next group around the best candidates. Keep the best policy found. Changing the weights can change the policy across many states at once.

The search uses at most as many training steps as the preceding Q-learning run. Both policies are evaluated on the same 20 held-out starts.


We score each weight vector $w$ by the mean return of its policy $\pi_w$
from the same three training starts $s_0^{(1)},s_0^{(2)},s_0^{(3)}$:

$$
\widehat J(w)=\frac{1}{3}\sum_{j=1}^{3}G(\pi_w;s_0^{(j)}),
$$

where $G(\pi_w;s_0^{(j)})$ is the total reward in that episode.

Starting from $\mu_0=0$ and $\sigma_0=1$, we sample up to 12 candidates
in each round $k$:

$$
w_{k,i}\sim\mathcal N(\mu_k,\sigma_k^2 I).
$$

Let $E_k$ index the three candidates with the highest scores (or all
candidates if fewer than three were sampled). We update the sampling
distribution by setting

$$
\mu_{k+1}=\frac{1}{|E_k|}\sum_{i\in E_k}w_{k,i},
\qquad
\sigma_{k+1}=\max\{0.15,\;0.8\,\sigma_k\}.
$$

The final policy uses the best weight vector tested across all rounds:
$$
w_{\mathrm{best}}\in\operatorname*{arg\,max}_{w\ \text{sampled}}\widehat J(w).
$$

In [ ]:
def acrobot_policy(weights):
    def act(state, rng):
        a, b, u, v = state
        features = (math.sin(a), math.cos(a), math.sin(b), math.cos(b), u/4, v/9)
        score = sum(w*x for w, x in zip(weights, features))
        return 2 if score > 0.10 else 0 if score < -0.10 else 1
    return act

def train_policy_search(env, steps, seed):
    if steps < 3 * env.limit:
        raise ValueError("Increase acro_episodes to allow three complete rollouts per candidate.")
    rng = Random(seed)
    start_seeds = [rng.randrange(1000000) for _ in range(3)]
    center = [0.0] * 6
    spread = 1.0
    best_score = float('-inf')
    best_policy = acrobot_policy(center)
    used = 0
    while used + 3 * env.limit <= steps:
        population = []
        for _ in range(12):
            if used + 3 * env.limit > steps:
                break
            weights = [rng.gauss(w, spread) for w in center]
            policy = acrobot_policy(weights)
            runs = [episode(env, policy, start) for start in start_seeds]
            used += sum(len(path) - 1 for path, _ in runs)
            score = sum(total for _, total in runs) / len(runs)
            population.append((score, weights))
            if score > best_score:
                best_score, best_policy = score, policy
        elite = sorted(population, key=lambda item: item[0], reverse=True)[:3]
        center = [sum(w[i] for _, w in elite) / len(elite) for i in range(6)]
        spread = max(0.15, 0.8 * spread)
    return best_policy, used


In [ ]:
acro_direct, policy_steps = train_policy_search(acro, acro_q_steps, acro_seed)
acro_results('Tabular Q-learning', acro_q, acro_q_steps)
acro_results('Direct policy search', acro_direct, policy_steps)


In [ ]:
replay(acro, acro_direct, acro_points, 'Acrobot · Direct policy search',
       ((-2.1, 2.1), (-2.1, 2.1)), seed=acro_replay_seed, target=1)
